# Вебинары курса Рекомендательные системы


# Вебинар 1. Введение, примеры задач, бизнес- и ML-метрики 

-----

# 1. Введение и примеры задач

RecSys everywhere

##  Постановка задачи

*Задача рекомендаций* - предложить пользователю товары, которые ему "понравятся"

Компоненты:
    - Пользователи (users)
    - Товары (items)
    - Взаимодействия (interactions) - по ним понимаем, что товар "понравился"

---

## Пользователи VS Товары


**Кейс**: определить в каждой ситуации, что является пользователем, а что - товаром
    
    1. На сайте Кинопоиска порекомендовать интересные фильмы
    2. На сайте hh.ru подобрать лучшие вакансии для соискателя вакансий
    3. На сайте hh.ru найти похожие вакансии на ту, которую сейчас просматривает пользователь ("вам также может понравиться")
    3. На сайте hh.ru подобрать лучших кандидатов для работодателя
    
*Пример ответа в чате:*
    1. Пользователь (П) - фильм, Товар (T) - фильм
    2. П - работодатель, Т - соискатель вакансий

**Ответ:**

1. В Кинопоиске пользователь — зритель, товар — фильм.
2. При подборе вакансий пользователь — соискатель, товар — вакансия.
3. Для похожих вакансий запросом служит открытая вакансия, кандидатами — другие вакансии. Если учитывать историю просмотров, рекомендации можно дополнительно персонализировать для соискателя.
4. При подборе кандидатов пользователь — работодатель, товар — резюме соискателя.

---

## Взаимодействия --> "понравился товар"



---

## В чем отличие от классификации?

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)


def _top_k(values, k):
    if not isinstance(k, (int, np.integer)) or k < 0:
        raise ValueError("k должен быть целым неотрицательным числом")
    return np.asarray(values)[:k]


def _prices(values, items):
    prices = np.asarray(values, dtype=float)
    if prices.ndim != 1 or len(prices) != len(items):
        raise ValueError("Каждому товару должна соответствовать одна цена")
    if not np.isfinite(prices).all() or (prices < 0).any():
        raise ValueError("Цены должны быть конечными и неотрицательными")
    return prices

In [2]:
pd.DataFrame([['Иван', 'Хлеб Бородинский', 1],
             ['Иван', 'Хлеб Белый', 0],
             ['Василий', 'Йогурт Epica', 1]], columns=['user', 'item', 'purchase_fact'])

,user,item,purchase_fact
0,Иван,Хлеб Бородинский,1
1,Иван,Хлеб Белый,0
2,Василий,Йогурт Epica,1


Ведь можно добавить фичи на user (средний чек, кол-во покупок в категориях и т.д), 
на item (цена, кол-во продаж в неделю и т.l), и решать задачу классификации. В чем отличие RecSys от классификации?

 - Много предсказаний для 1 user (extreme classification)
 - Гораздо больший объем данных: 100K users, 10K items --> 1B предсказаний
 - Большинство товаров user никогда не видел --> Не взаимодействовал --> 0 не значит "не понравился"
 - Нет явного таргета. Не понятно, что значит "(не) понравился"
 - Feedback loop
 - Иногда важен порядок рекомендаций

In [3]:
pd.DataFrame([['Иван', 'Хлеб Бородинский', 1],
              ['Иван', 'Хлеб Белый', 0],
              ['Иван', 'Йогурт Epica', "?"],
              ['Василий', 'Хлеб Бородинский', "?"],
              ['Василий', 'Хлеб Белый', "?"],
              ['Василий', 'Йогурт Epica', 1]], columns=['user', 'item', 'purchase_fact'])

,user,item,purchase_fact
0,Иван,Хлеб Бородинский,1
1,Иван,Хлеб Белый,0
2,Иван,Йогурт Epica,?
3,Василий,Хлеб Бородинский,?
4,Василий,Хлеб Белый,?
5,Василий,Йогурт Epica,1


----

## Основные типы задач:
    
**Рекомендация топ-К товаров**: Дан список товаров. Рекомендовать пользователю K товаров, которые ему понравятся
    - e-mail рассылки (М.Видео, Hoff, Пятерочка)
    - push-уведомления (Delivery Club, HeadHunter, Виктория)
    - Рекомендации в отдельном окне на сайте (vk, okko)

**Ранжирование товаров**: Дан список товаров. Нужно его отранжировать в порядке убывания интереса для пользователя
    - Ранжирование каталога товаров (Wildberries, Lamoda)
    - Ранжирование ленты (vk, Я.Дзен)
    - Ранжирование поисковой выдачи (Яндекс, Гугл)
    - Ранжирование "каруселей" товаров (Delivery Club, Я.Еда)
    
**Поиск похожих товаров**: Дан 1 товар. Нужно найти наиболее похожие на него товары
    - "Вам также может понравиться" (Ozon, Циан)
    - Похожим пользователям понравилось (vk)
    - Вы можете быть знакомы (vk, facebook)
    
**Рекомендация дополнителдьного товара**. Дан 1 товар. Найти товары, которые покупают вместе с этим товаром
    - С этим товаром часто покупают (Я.Маркет, Ozon, М.Видео)
    


---

# 2. ML-mетрики качества

*Хороши ли рекомендации, предлагаемые моделью?*  
Обычно считаются для каждого юзера, затем усредняются по юзерам

Представим, что рекомендовано всего 10 товаров
Куплено - 4 товара, некоторые не из рекомендованного списка

In [4]:
recommended_list = [143, 156, 1134, 991, 27, 1543, 3345, 533, 11, 43] #id товаров
bought_list = [521, 32, 143, 991]

In [5]:
len(recommended_list)

10

### 1. Hit rate

Hit rate = был ли хотя бы 1 релевантный товар среди рекомендованных

- Иногда применяется, когда продаются достаточно дрогие товары (например, бытовая техника) 

----
Hit rate = (был ли хотя бы 1 релевантный товар среди рекомендованных)   

Hit rate@k = (был ли хотя бы 1 релевантный товар среди топ-k рекомендованных)

In [6]:
def hit_rate(recommended_list, bought_list):
    return int(np.isin(recommended_list, bought_list).any())


def hit_rate_at_k(recommended_list, bought_list, k=5):
    # your_code
    recommended = _top_k(recommended_list, k)
    return hit_rate(recommended, bought_list)

In [7]:
hit_rate(recommended_list, bought_list)

1

### 2. Precision

*Precision* - доля релевантных товаров среди рекомендованных = Какой % рекомендованных товаров  юзер купил

- Пожалуй, самая приближенная к бизнес-метрикам и самая популярная метрика

---

Precision= (# of recommended items that are relevant) / (# of recommended items)  

Precision@k = (# of recommended items @k that are relevant) / (# of recommended items @k)

Money Precision@k = (revenue of recommended items @k that are relevant) / (revenue of recommended items @k)  

**Note:** Обычно k в precision@k достаточно невелико (5-20) и определяется из бизнес-логики. Например, 5 товаров в e-mail рассылке, 20 ответов на первой странице google и т.д

Красная рыба - 400 руб  
Молоко - 60 руб  
Хлеб = 40 руб  
Гречка = 40 руб  
Шоколад = 90 руб  

------  
Варенье - 240 руб  
...  

**Case 1** 
рекомендовали - Красная рыба, Молоко, Хлеб, Гречка, Шоколад 
купили из рекомендованного 1 и последний товар


prices_reсommended = [400, 60, 40, 40 , 90]  
flags = [1, 0, 0, 0 , 1]  

$precison@5 = \frac{1 + 0 + 0 +0 + 1}{1+1+1+1+1} = 40\%$  
$money precision@5 = \frac{1*400 + 0*60 + ... + 1*90}{1*400 + 1*60 + ... + 1*90} = 77.7\%$  

  
**Case 2**   
рекомендовали - Молоко, Красная рыба,  Хлеб, Гречка, Шоколад 
купили из рекомендованного 1 и последний товар


prices_reсommended = [60, 400,  40, 40 , 90]  
flags = [1, 0,  0, 0 , 1]  

$precison@5 = \frac{0 + 1 + 0 +0 + 1}{1+1+1+1+1} = 40\%$  
$money precision@5 = \frac{0*400 + 1*60 + ... + 1*90}{1*400 + 1*60 + ... + 1*90} = 23.8\%$

In [8]:
def precision(recommended_list, bought_list):
    recommended = np.asarray(recommended_list)
    return float(np.isin(recommended, bought_list).mean()) if recommended.size else 0.0


def precision_at_k(recommended_list, bought_list, k=5):
    recommended = _top_k(recommended_list, k)
    return precision(recommended, bought_list)


def money_precision_at_k(recommended_list, bought_list, prices_recommended, k=5):
    # your_code
    # Лучше считать через скалярное произведение, а не цикл
    prices = _prices(prices_recommended, recommended_list)
    recommended = _top_k(recommended_list, k)
    prices = prices[:len(recommended)]
    flags = np.isin(recommended, bought_list)
    total = prices.sum()
    return float(prices @ flags / total) if total else 0.0

In [9]:
precision(recommended_list, bought_list)

0.2

In [10]:
precision_at_k(recommended_list, bought_list, k=5)

0.4

In [11]:
precision_at_k(recommended_list, bought_list, k=3)

0.3333333333333333

### 3. Recall

*Recall* - доля рекомендованных товаров среди релевантных = Какой % купленных товаров был среди рекомендованных

- Обычно используется для моделей пре-фильтрации товаров (убрать те товары, которые точно не будем рекомендовать)

---

Recall= (# of recommended items that are relevant) / (# of relevant items)  

Recall@k = (# of recommended items @k that are relevant) / (# of relevant items)

Money Recall@k = (revenue of recommended items @k that are relevant) / (revenue of relevant items)  

    
  
**Note:** в recall@k число k обычно достаточно большое (50-200), больше чем покупок у среднестатистического юзера

In [12]:
def recall(recommended_list, bought_list):
    bought = np.unique(bought_list)
    return float(np.isin(bought, recommended_list).mean()) if bought.size else 0.0


def recall_at_k(recommended_list, bought_list, k=5):
    # your_code
    recommended = _top_k(recommended_list, k)
    return recall(recommended, bought_list)


def money_recall_at_k(recommended_list, bought_list, prices_recommended, prices_bought, k=5):
    # your_code
    recommended_prices = _prices(prices_recommended, recommended_list)
    bought_prices = _prices(prices_bought, bought_list)
    recommended = _top_k(recommended_list, k)
    flags = np.isin(recommended, bought_list)
    revenue = recommended_prices[:len(recommended)] @ flags
    total = bought_prices.sum()
    return float(revenue / total) if total else 0.0

In [13]:
# Проверяем обычные и денежные метрики на примере с известными ценами.
products = [1, 2, 3, 4, 5]
purchases = [1, 5, 6]
prices = [400, 60, 40, 40, 90]
purchase_prices = [400, 90, 240]

assert hit_rate_at_k(products, purchases, 5) == 1
assert hit_rate_at_k(products, purchases, 0) == 0
assert np.isclose(precision_at_k(products, purchases, 5), 2 / 5)
assert np.isclose(recall_at_k(products, purchases, 5), 2 / 3)
assert np.isclose(money_precision_at_k(products, purchases, prices), 490 / 630)
assert np.isclose(money_recall_at_k(products, purchases, prices, purchase_prices), 490 / 730)
assert precision([], purchases) == recall(products, []) == 0
assert money_precision_at_k([], purchases, []) == 0
assert money_recall_at_k(products, [], prices, []) == 0

pd.Series({
    'hit_rate@5': hit_rate_at_k(products, purchases),
    'precision@5': precision_at_k(products, purchases),
    'recall@5': recall_at_k(products, purchases),
    'money_precision@5': money_precision_at_k(products, purchases, prices),
    'money_recall@5': money_recall_at_k(products, purchases, prices, purchase_prices),
})

hit_rate@5           1.000000
precision@5          0.400000
recall@5             0.666667
money_precision@5    0.777778
money_recall@5       0.671233
dtype: float64

Списки содержат уникальные товары, каждому соответствует одна цена. В денежном recall цены одного товара в рекомендациях и покупках считаем одинаковыми. При пустом списке или нулевой сумме цен возвращаем 0. Если k больше длины списка, считаем метрику по всем выданным рекомендациям.

In [14]:
recall(recommended_list, bought_list)

0.5

---

# Метрики ранжирования

Если важен порядок рекомендаций. Подробнее можно почитать [здесь](https://habr.com/ru/company/econtenta/blog/303458/). Формулы в статье могут несколько отличаться от формул в лекции 

## AP@k
AP@k - average precision at k

$$AP@k = \frac{1}{r} \sum{[recommended_{relevant_i}] * precision@k}$$

- r - кол-во релевантный среди рекомендованных
- Суммируем по всем релевантным товарам
- Зависит от порядка реокмендаций

In [15]:
def ap_k(recommended_list, bought_list, k=5):
    recommended = _top_k(recommended_list, k)
    flags = np.isin(recommended, bought_list)
    relevant = flags.sum()
    if relevant == 0:
        return 0.0
    precisions = np.cumsum(flags) / np.arange(1, len(flags) + 1)
    # В этой лекции r — число релевантных товаров именно среди top-k.
    return float(precisions @ flags / relevant)

In [16]:
ap_k(recommended_list, bought_list, k=5)

0.75

### MAP@k

MAP@k (Mean Average Precision@k)  
Среднее AP@k по всем юзерам
- Показывает средневзвешенную точность рекомендаций

$$MAP@k = \frac{1}{|U|} \sum_u{AP_k}$$
  
|U| - кол-во юзеров

In [17]:
recommended_list = [[143, 156, 1134, 991, 27, 1543, 3345, 533, 11, 43],
                    [146, 156, 1134, 991, 27, 1543, 3345, 533, 11, 43],] #id товаров
bought_list = [[521, 32, 143, 991], [146, 29]]

In [18]:
def map_k(recommended_list, bought_list, k=5, u=None):
    # your_code
    if len(recommended_list) != len(bought_list):
        raise ValueError("Для каждого пользователя нужны рекомендации и покупки")
    if u is not None and u != len(recommended_list):
        raise ValueError("u должно совпадать с числом переданных пользователей")
    scores = [ap_k(recs, bought, k) for recs, bought in zip(recommended_list, bought_list)]
    return float(np.mean(scores)) if scores else 0.0

In [19]:
# Включаем в сумму попадание на последней позиции k.
assert np.isclose(ap_k([1, 9, 2], [2], k=3), 1 / 3)
assert ap_k([1, 2], [], k=5) == 0
assert ap_k([], [1], k=5) == 0
assert np.isclose(map_k([[1, 9], [9, 1]], [[1], [1]], k=2), 0.75)
map_k(recommended_list, bought_list, k=5)

0.875

Здесь AP@k следует формуле лекции: знаменатель — число попаданий среди первых k рекомендаций. В lab02 задан другой знаменатель, min(k, n_u), поэтому численные значения AP между этими лабами напрямую не сравниваем.

### AUC@k
AUC для первых k наблюдений  
- Можно посчитать как sklern.metrics.roc_auc_score для топ-k предсказаний
- Показывает долю верно отранжированных товаров

In [20]:
from sklearn.metrics import roc_auc_score

# Скоры убывают с позицией в выдаче, нужны оба класса релевантности.
auc_recs = np.asarray(recommended_list[0][:5])
auc_labels = np.isin(auc_recs, bought_list[0]).astype(int)
auc_scores = np.arange(len(auc_recs), 0, -1)
roc_auc_score(auc_labels, auc_scores)

0.6666666666666667

### NDCG@k
Normalized discounted cumulative gain

$DCG = \frac{1}{|r|} \sum_u{\frac{[bought fact]}{discount(i)}}$  

$discount(i) = 1$ if $i <= 2$,   
$discount(i) = log_2(i)$ if $i > 2$


(!) Считаем для первых k рекоммендаций   
(!) - существуют вариации с другими $discount(i)$  
i - ранк рекомендованного товара  
|r| - кол-во рекомендованных товаров 

$$NDCG = \frac{DCG}{ideal DCG}$$


https://pytorch.org/docs/stable/index.html

In [21]:
import torch

torch.manual_seed(42)
N = 5
ys_true = torch.randint(0, 5, (N, ))
ys_pred = torch.rand(N)

In [22]:
ys_true

tensor([2, 2, 1, 4, 1])

In [23]:
ys_pred

tensor([0.6009, 0.2566, 0.7936, 0.9408, 0.1332])

In [24]:
# итоговый результат модели - отсортированный ys_pred
ys_pred

tensor([0.6009, 0.2566, 0.7936, 0.9408, 0.1332])

In [25]:
ys_pred = torch.as_tensor(ys_pred, dtype=torch.float32)

In [26]:
ys_pred

tensor([0.6009, 0.2566, 0.7936, 0.9408, 0.1332])

In [27]:
def compute_gain(y_value: float, gain_scheme: str) -> float:
    value = float(y_value)
    if gain_scheme == "exp2":
        gain = 2 ** value - 1
    elif gain_scheme == "const":
        gain = value
    else:
        raise ValueError(f"{gain_scheme} method not supported, only exp2 and const.")
    return float(gain)

In [28]:
def dcg(ys_true: torch.Tensor, ys_pred: torch.Tensor, gain_scheme: str) -> float:
    ys_true = torch.as_tensor(ys_true, dtype=torch.float64)
    ys_pred = torch.as_tensor(ys_pred, dtype=torch.float64)
    if ys_true.ndim != 1 or ys_true.shape != ys_pred.shape:
        raise ValueError("Релевантности и скоры должны быть векторами одинаковой длины")
    if not torch.isfinite(ys_true).all() or (ys_true < 0).any() or not torch.isfinite(ys_pred).all():
        raise ValueError("Релевантности должны быть неотрицательными, значения — конечными")
    if gain_scheme not in {"const", "exp2"}:
        raise ValueError("Поддерживаются схемы const и exp2")
    _, argsort = torch.sort(ys_pred, descending=True, dim=0, stable=True)
    ys_true_sorted = ys_true[argsort]
    ret = 0.0
    for idx, cur_y in enumerate(ys_true_sorted, 1):
        gain = compute_gain(cur_y, gain_scheme)
        # your code
        discount = 1.0 if idx <= 2 else np.log2(idx)
        ret += gain / discount
    return ret / len(ys_true) if len(ys_true) else 0.0

In [29]:
_, argsort = torch.sort(ys_pred, descending=True, dim=0)

In [30]:
ys_pred

tensor([0.6009, 0.2566, 0.7936, 0.9408, 0.1332])

In [31]:
argsort

tensor([3, 2, 0, 1, 4])

In [32]:
ys_true_sorted = ys_true[argsort]

In [33]:
for idx, cur_y in enumerate(ys_true_sorted, 1):
    print(idx, cur_y)

1 tensor(4)
2 tensor(1)
3 tensor(2)
4 tensor(2)
5 tensor(1)


In [34]:
def ndcg(ys_true: torch.Tensor, ys_pred: torch.Tensor, gain_scheme: str = 'const') -> float:
    pred_dcg = dcg(ys_true, ys_pred, gain_scheme)
    # your code ideal_dcg = 
    ideal_dcg = dcg(ys_true, ys_true, gain_scheme)
    return pred_dcg / ideal_dcg if ideal_dcg else 0.0

In [35]:
# Дисконт из задания: 1 на первых двух позициях, затем log2(rank).
relevance = torch.tensor([3., 0., 1.])
scores = torch.tensor([0.9, 0.8, 0.7])
expected_dcg = (3 + 1 / np.log2(3)) / 3
assert np.isclose(dcg(relevance, scores, 'const'), expected_dcg)
for scheme in ('const', 'exp2'):
    assert np.isclose(ndcg(relevance, relevance, scheme), 1.0)
    assert 0 <= ndcg(relevance, scores, scheme) <= 1
assert ndcg(torch.zeros(3), scores) == 0
assert ndcg(torch.tensor([]), torch.tensor([])) == 0

pd.Series({scheme: ndcg(ys_true, ys_pred, scheme) for scheme in ('const', 'exp2')})

const    0.938969
exp2     0.951977
dtype: float64

### MRR@k
Mean Reciprocal Rank

- Считаем для первых k рекоммендаций
- Найти ранк первого релевантного предсказания $k_u$
- Посчитать reciprocal rank = $\frac{1}{k_u}$

$$MRR = mean(\frac{1}{k_u})$$

[MRR](https://drive.google.com/file/d/1uMFKqWULFmehS0pJtvXKRygjYxQOPise/view?usp=sharing)

In [36]:
def reciprocal_rank(recommended_list, bought_list):
    # your_code
    positions = np.flatnonzero(np.isin(recommended_list, bought_list))
    return float(1.0 / (positions[0] + 1)) if positions.size else 0.0

In [37]:
assert reciprocal_rank([8, 7, 3], [3, 6]) == 1 / 3
assert reciprocal_rank([8, 7], [3]) == 0
assert reciprocal_rank([], [3]) == 0
rr_values = [reciprocal_rank(recs[:5], bought) for recs, bought in zip(recommended_list, bought_list)]
pd.Series({'RR первого пользователя': rr_values[0],
           'RR второго пользователя': rr_values[1],
           'MRR@5': np.mean(rr_values)})

RR первого пользователя    1.0
RR второго пользователя    1.0
MRR@5                      1.0
dtype: float64

# 3. Связь бизнес-метрик, ML-метрик и функции потерь(loss)

- **loss** - то, что оптимизирует модель (RMSE - Root Mean Squared Error)
- **ML-метрика** - то, как мы измеряем качество на test (money precision@5)
- **Бизнес-метрика** - то, что хочет оптимизировать бизнес (выручка)

В идеале loss = ML-метрика = бизнес-метрика, но это возмонжо в очень редких случаях

Надеемся, что:  
    *Оптимизация loss --> рост ML-метрик --> рост бизнес-метрик*

### Пример: рекомендательная система товаров на сайте  
      
На сайте есть баннер, на котором мы можем разместить 5 товаров. Надо порекомендовать эти 5 товаров персонально каждому юзеру

*Шаг 1: Определим бизнес-метрику*  
Бизнес хочет максимизировать выручку --> бизнес-метрика - **Выручка**  

*Шаг 2: Разложим ее на составляющие*  
Выручка =   
       Средний чек * кол-во покупок =   
       Средний чек * (число юзеров * конверсия из захода на сайт в заказ) =
       Число юзеров * (Средний чек * конверсия из захода на сайт в заказ)
       
Рекомендательная система влияет только на (Средний чек * конверсия из захода на сайт в заказ).   
Хорошим приближением этого является **money precision@5** - ML-метрика

*Шаг 3: loss*  
Это существенно сложнее. Персонализированные ML-модели не умеют напрямую оптимизировать конверсию. Стандартный loss - **RMSE**. Можно для начала попробовать его. 

P.S. Если вы хотите приблизить RMSE к money precision@5, то можно посчитать weighted RMSE, где вес каждого наблюдения = стоимость товара.   
P.S.S. Про другие виды loss будет рассказано в курсе